# 5.9 · XGBoost

> **课程定位 / Where this fits**
> 5.8 的 GBDT 是地基。XGBoost(eXtreme Gradient Boosting)在它之上做了一系列**数学 + 工程**改进: 二阶泰勒近似、叶子权重正则化、列采样、缺失值处理、并行化。它是 2015–2020 年 Kaggle 表格赛的绝对霸主, 至今仍是工业界首选之一。
> XGBoost adds a second-order Taylor objective, leaf-weight regularization, column subsampling, missing-value handling, and parallelism on top of GBDT. The long-reigning tabular champion.

> 💡 **面试相关 / Interview-relevant**
> - "XGBoost 比传统 GBDT 强在哪" ★★★★★（二阶+正则+工程）
> - "XGBoost 的目标函数 / 为什么用二阶泰勒" ★★★★★
> - "叶子权重的正则项 λ、γ 各是什么" ★★★★
> - "XGBoost 怎么处理缺失值" ★★★★（默认方向）
> - "重要超参数有哪些" ★★★★
> - "XGBoost vs LightGBM" ★★★★（5.10 对比）

---

## 学习目标 / Learning Objectives
1. XGBoost 的**正则化目标函数** + 二阶泰勒展开。
2. 由此推出**叶子最优权重**与**分裂增益**公式。
3. 关键超参数与调参直觉。
4. 缺失值默认方向、早停、特征重要性。
5. 与 sklearn GBDT 的对比。

## 目录 / TOC
1. [正则化目标 + 二阶泰勒 ⭐](#1)
2. [叶子权重与分裂增益 ⭐](#2)
3. [💰 数据 + 训练](#3)
4. [关键超参数 ⭐](#4)
5. [缺失值 + 早停 + 重要性](#5)
6. [小结](#6)


<a id="1"></a>
## 1. 正则化目标 + 二阶泰勒 ⭐ / Regularized Objective & 2nd-order Taylor

XGBoost 的目标 = 损失 + **树复杂度正则**:
$$\mathcal{L} = \sum_i l(y_i, \hat y_i) + \sum_m \Omega(f_m), \qquad \Omega(f) = \gamma T + \tfrac12\lambda\|\mathbf{w}\|^2$$
其中 $T$ = 叶子数, $\mathbf{w}$ = 叶子权重向量, $\gamma$ 惩罚叶子数(剪枝), $\lambda$ 是叶子权重的 L2(像 4.4 岭回归)。

加第 $m$ 棵树 $f_m$ 时, 对损失做**二阶泰勒展开**(GBDT 只用一阶):
$$\mathcal{L}^{(m)} \approx \sum_i\Big[g_i f_m(\mathbf{x}_i) + \tfrac12 h_i f_m(\mathbf{x}_i)^2\Big] + \Omega(f_m)$$
其中 $g_i=\partial_{\hat y} l$(一阶梯度), $h_i=\partial^2_{\hat y} l$(二阶, Hessian)。二阶信息让每步更新更精确、收敛更快——这是 XGBoost 的核心数学改进。


<a id="2"></a>
## 2. 叶子权重与分裂增益 ⭐ / Leaf Weight & Split Gain

固定树结构, 对叶子 $j$(样本集合 $I_j$)求最优权重(令导数=0):
$$w_j^* = -\frac{\sum_{i\in I_j} g_i}{\sum_{i\in I_j} h_i + \lambda}, \qquad \mathcal{L}^* = -\frac12\sum_j \frac{(\sum_{i\in I_j} g_i)^2}{\sum_{i\in I_j} h_i + \lambda} + \gamma T$$

由此得**分裂增益**(分裂前后目标改善):
$$\text{Gain} = \frac12\Big[\frac{G_L^2}{H_L+\lambda} + \frac{G_R^2}{H_R+\lambda} - \frac{(G_L+G_R)^2}{H_L+H_R+\lambda}\Big] - \gamma$$
其中 $G=\sum g, H=\sum h$。树按 Gain 选分裂; **Gain < 0(被 $\gamma$ 拉负)就不分裂** → 内置剪枝。这套封闭式是 XGBoost 又快又准的关键。


<a id="3"></a>
## 3. 数据 + 训练 / Data & Training

复用 5.8 的**合成 Adult Income**(同一个生成器, 便于和 GBDT/LightGBM/CatBoost 横向比较)。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
sns.set_theme(style="whitegrid")
print("xgboost", xgb.__version__)

def make_income(n=8000, seed=0):
    rng = np.random.default_rng(seed)
    age = rng.integers(18, 70, n); edu_years = rng.integers(6, 21, n)
    hours = rng.normal(40, 10, n).clip(10, 80)
    capital_gain = (rng.random(n) < 0.15) * rng.exponential(5000, n)
    logit = (-9 + 0.04*age - 0.0004*(age-45)**2 + 0.25*edu_years + 0.02*hours
             + 0.0002*np.sqrt(capital_gain)*edu_years*0.3 + rng.normal(0, 0.5, n))
    y = (rng.random(n) < 1/(1+np.exp(-logit))).astype(int)
    X = pd.DataFrame({"age": age, "edu_years": edu_years, "hours": hours,
                      "capital_gain": capital_gain.round(0)})
    return X, y

X, y = make_income()
print(f"合成 Adult Income: {X.shape}, 高收入率 {y.mean():.0%}")
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)


In [ ]:
clf = xgb.XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=4,
    reg_lambda=1.0, gamma=0.0, subsample=0.8, colsample_bytree=0.8,
    eval_metric="auc", random_state=0)
clf.fit(X_tr, y_tr)
print(f"XGBoost test 准确率: {accuracy_score(y_te, clf.predict(X_te)):.3f}")
print(f"XGBoost test AUC:    {roc_auc_score(y_te, clf.predict_proba(X_te)[:,1]):.3f}")

from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=0).fit(X_tr, y_tr)
print(f"对照 sklearn GBDT AUC: {roc_auc_score(y_te, gb.predict_proba(X_te)[:,1]):.3f}")


<a id="4"></a>
## 4. 关键超参数 ⭐ / Key Hyperparameters

| 参数 | 控制 | 调参方向 |
|---|---|---|
| `n_estimators` + `learning_rate` | 树数 × 步长 | 小 lr + 多树 + 早停 |
| `max_depth` | 树深(交互阶数) | 3–8, 大→过拟合 |
| `reg_lambda` (λ) | 叶子权重 L2 | 大→更平滑 |
| `gamma` (γ) | 分裂最小增益(剪枝) | 大→更保守 |
| `subsample` | 行采样 | <1 加随机性防过拟合 |
| `colsample_bytree` | 列采样 | <1 去相关(借鉴 RF) |
| `min_child_weight` | 叶子最小 Hessian 和 | 大→更保守 |


In [ ]:
# max_depth 与 reg_lambda 的过拟合控制 / regularization sweep
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for depth in [2, 4, 8]:
    tr, te = [], []
    for nt in [20, 50, 100, 200, 300]:
        m = xgb.XGBClassifier(n_estimators=nt, max_depth=depth, learning_rate=0.1,
                              eval_metric="auc", random_state=0).fit(X_tr, y_tr)
        tr.append(roc_auc_score(y_tr, m.predict_proba(X_tr)[:,1]))
        te.append(roc_auc_score(y_te, m.predict_proba(X_te)[:,1]))
    axes[0].plot([20,50,100,200,300], te, "o-", label=f"depth={depth}")
axes[0].set_xlabel("n_estimators"); axes[0].set_ylabel("test AUC"); axes[0].legend()
axes[0].set_title("max_depth 越大越易过拟合")

lambdas = [0, 0.1, 1, 10, 100]
te_l = [roc_auc_score(y_te, xgb.XGBClassifier(n_estimators=200, max_depth=6, reg_lambda=lam,
        eval_metric="auc", random_state=0).fit(X_tr,y_tr).predict_proba(X_te)[:,1]) for lam in lambdas]
axes[1].semilogx([0.05]+lambdas[1:], te_l, "s-")
axes[1].set_xlabel("reg_lambda (λ)"); axes[1].set_ylabel("test AUC")
axes[1].set_title("reg_lambda: 叶子权重 L2 正则")
plt.tight_layout(); plt.show()


<a id="5"></a>
## 5. 缺失值 + 早停 + 重要性 / Missing, Early Stop, Importance

**缺失值**: XGBoost 为每个分裂学一个**默认方向**——缺失的样本自动走该方向, 无需填补(对比 3.2 需手动填)。


In [ ]:
# 缺失值原生处理 / native NaN handling
X_miss = X_tr.copy()
rng = np.random.default_rng(1)
mask = rng.random(X_miss.shape) < 0.1
X_miss = X_miss.mask(mask)     # 10% 设为 NaN
print(f"训练集注入 {mask.mean():.0%} 缺失; XGBoost 直接训练(学默认方向):")
m = xgb.XGBClassifier(n_estimators=200, max_depth=4, eval_metric="auc", random_state=0).fit(X_miss, y_tr)
print(f"  含缺失训练 test AUC: {roc_auc_score(y_te, m.predict_proba(X_te)[:,1]):.3f} (无需填补)")

# 早停 / early stopping
clf_es = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=4,
                           eval_metric="auc", early_stopping_rounds=20, random_state=0)
clf_es.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)
print(f"\n早停: 上限1000, 实际最佳迭代 {clf_es.best_iteration} 棵")

# 特征重要性 / importance
imp = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\n特征重要性(gain):"); print(imp.round(3).to_string())


<a id="6"></a>
## 6. 小结 / Summary

```
XGBoost = GBDT + 二阶泰勒 + 正则化 + 工程优化
目标: Σl(y,ŷ) + Σ[γT + ½λ‖w‖²]; 二阶展开用 g(梯度)+h(Hessian)
叶子最优权重 w* = -Σg/(Σh+λ); 分裂增益含 G²/(H+λ), Gain<γ 不分裂(剪枝)
关键超参: lr×n_estimators, max_depth, reg_lambda(λ), gamma(γ), subsample, colsample
缺失值: 学默认方向, 无需填补; 早停防过拟合
```

### 💡 面试速查
1. **比 GBDT 强**: 二阶泰勒(更准更快) + 叶子正则(λ,γ) + 列采样 + 缺失处理 + 并行
2. **目标函数**: 损失 + γT + ½λ‖w‖²; 最优叶权 -Σg/(Σh+λ)
3. **γ** 惩罚叶子数(剪枝), **λ** 是叶权 L2
4. **缺失值**走学到的默认方向, 不用填补
5. 小 lr + 多树 + 早停 + 行列采样是标准配方

### 下一节
**5.10 LightGBM**——微软的 boosting。用直方图分箱 + 按叶生长(leaf-wise) + GOSS/EFB, 比 XGBoost 更快, 大数据首选。
